In [2]:
"""
fix_trial_codes.py
Convert fake trial codes (CODE_XXXX) to real MIMIC codes using text matching.
"""

import os
import json
import pandas as pd
from config import Config

def load_mimic_diagnosis_mapping(cfg):
    """Load diagnosis descriptions and map them to ICD-10 codes."""
    # Try to load the diagnosis descriptions file
    desc_path = f"{cfg.DATA_DIR}/D_ICD_DIAGNOSES.csv"
    
    if not os.path.exists(desc_path):
        print(f"⚠️ Diagnosis descriptions not found at {desc_path}")
        print("   Using fallback mapping...")
        return {}
    
    desc_df = pd.read_csv(desc_path)
    desc_df.columns = [c.upper() for c in desc_df.columns]
    
    # Clean ICD-9 codes
    desc_df['ICD9_CODE'] = desc_df['ICD9_CODE'].astype(str).str.replace('.', '').str.strip()
    desc_df['LONG_TITLE'] = desc_df['LONG_TITLE'].str.lower().str.strip()
    desc_df['SHORT_TITLE'] = desc_df['SHORT_TITLE'].str.lower().str.strip()
    
    # Create mapping from description to ICD-10 code
    # First, load the ICD-9 to ICD-10 mapping from preprocessor
    from preprocessor import MIMICDataPreprocessor
    pp = MIMICDataPreprocessor(cfg)
    icd9_to_icd10 = pp.icd9_to_icd10_map
    
    # Create mapping
    text_to_code = {}
    for _, row in desc_df.iterrows():
        icd9 = row['ICD9_CODE']
        icd10 = icd9_to_icd10.get(icd9, icd9)
        
        long_title = row['LONG_TITLE']
        short_title = row['SHORT_TITLE']
        
        if long_title and long_title != 'nan':
            text_to_code[long_title] = icd10
        if short_title and short_title != 'nan':
            text_to_code[short_title] = icd10
    
    print(f"✅ Loaded {len(text_to_code)} diagnosis descriptions")
    return text_to_code

def fix_trial_codes(trials, text_to_code):
    """Replace fake codes with real ones using text matching."""
    
    fixed_trials = []
    total_fixed = 0
    total_criteria = 0
    
    for trial in trials:
        fixed_criteria = []
        
        for c in trial.get('criteria', []):
            total_criteria += 1
            raw_text = c.get('raw_entity', '').lower()
            
            # Skip empty criteria
            if not raw_text:
                fixed_criteria.append(c)
                continue
            
            # Try to find matching code from text
            matched_code = None
            for desc, code in text_to_code.items():
                if raw_text in desc or desc in raw_text:
                    matched_code = code
                    break
            
            if matched_code:
                # Update the criterion with the real code
                c['entity_code'] = matched_code
                total_fixed += 1
            
            fixed_criteria.append(c)
        
        fixed_trials.append({
            'nct_id': trial.get('nct_id'),
            'title': trial.get('title', ''),
            'conditions': trial.get('conditions', []),
            'phase': trial.get('phase', 'NA'),
            'sample_size': trial.get('sample_size', 100),
            'criteria': fixed_criteria
        })
    
    if total_criteria > 0:
        print(f"✅ Fixed {total_fixed} out of {total_criteria} criteria ({total_fixed/total_criteria*100:.2f}%)")
    else:
        print("⚠️ No criteria found to fix")
    
    return fixed_trials

def check_codes_before_after(trials, text_to_code):
    """Show sample of codes before and after fixing."""
    sample_trial = trials[0] if trials else None
    if not sample_trial:
        return
    
    print("\n📋 Sample code conversion:")
    for c in sample_trial.get('criteria', [])[:5]:
        raw_text = c.get('raw_entity', '')[:40]
        old_code = c.get('entity_code')
        
        # Find what it would map to
        new_code = None
        for desc, code in text_to_code.items():
            if raw_text.lower() in desc or desc in raw_text.lower():
                new_code = code
                break
        
        if new_code:
            print(f"   '{raw_text}...' -> {old_code} → {new_code} ✅")

def main():
    cfg = Config()
    
    # Load trials
    train_path = f"{cfg.TRIALS_DATA_DIR}/structured_clinical_trials.json"
    eval_path = f"{cfg.TRIALS_DATA_DIR}/structured_clinical_trials_eval.json"
    
    # Check if files exist
    if not os.path.exists(train_path):
        print(f"❌ Training trials not found at {train_path}")
        # Try alternate location
        alt_path = "structured_clinical_trials.json"
        if os.path.exists(alt_path):
            print(f"✅ Found trials at {alt_path}")
            train_path = alt_path
        else:
            print("❌ No trial files found!")
            return
    
    if not os.path.exists(eval_path):
        print(f"⚠️ Evaluation trials not found at {eval_path}, using only training trials")
        eval_trials = []
    else:
        with open(eval_path, 'r') as f:
            eval_trials = json.load(f)
    
    with open(train_path, 'r') as f:
        train_trials = json.load(f)
    
    print(f"Loaded {len(train_trials)} training trials, {len(eval_trials)} eval trials")
    
    # Load diagnosis mapping
    text_to_code = load_mimic_diagnosis_mapping(cfg)
    
    if not text_to_code:
        print("\n⚠️ No diagnosis mapping available. Using fallback text matching...")
        # Create a simple fallback mapping
        text_to_code = {
            'heart failure': 'I509',
            'myocardial infarction': 'I219',
            'diabetes': 'E119',
            'pneumonia': 'J189',
            'sepsis': 'A419',
            'atrial fibrillation': 'I480',
            'hypertension': 'I10',
            'chronic kidney disease': 'N189',
            'stroke': 'I639',
            'cancer': 'C809',
        }
        print(f"   Using {len(text_to_code)} fallback mappings")
    
    # Show sample before fixing
    print("\n📋 Sample codes BEFORE fixing:")
    for c in train_trials[0].get('criteria', [])[:3]:
        print(f"   {c.get('raw_entity', '')[:30]:30} -> {c.get('entity_code')}")
    
    # Fix codes
    print("\n🔧 Fixing training trials...")
    fixed_train = fix_trial_codes(train_trials, text_to_code)
    
    if eval_trials:
        print("\n🔧 Fixing evaluation trials...")
        fixed_eval = fix_trial_codes(eval_trials, text_to_code)
    else:
        fixed_eval = []
    
    # Show sample after fixing
    print("\n📋 Sample codes AFTER fixing:")
    for c in fixed_train[0].get('criteria', [])[:3]:
        print(f"   {c.get('raw_entity', '')[:30]:30} -> {c.get('entity_code')}")
    
    # Save fixed trials
    with open(train_path, 'w') as f:
        json.dump(fixed_train, f, indent=2)
    print(f"\n✅ Saved {len(fixed_train)} fixed training trials to {train_path}")
    
    if fixed_eval:
        with open(eval_path, 'w') as f:
            json.dump(fixed_eval, f, indent=2)
        print(f"✅ Saved {len(fixed_eval)} fixed evaluation trials to {eval_path}")
    
    # Also save to current directory for backward compatibility
    with open('structured_clinical_trials.json', 'w') as f:
        json.dump(fixed_train, f, indent=2)
    if fixed_eval:
        with open('structured_clinical_trials_eval.json', 'w') as f:
            json.dump(fixed_eval, f, indent=2)
    print("✅ Also saved to current directory")

if __name__ == "__main__":
    main()

Loaded 149 training trials, 38 eval trials


✅ Loaded 27246 diagnosis descriptions

📋 Sample codes BEFORE fixing:
   * Participant must be ≥ 28 day -> CODE_8730
   * Body weight ≥ 5 kg at the ti -> CODE_8239
   * Documented TMA on/after 01 J -> CODE_4891

🔧 Fixing training trials...
✅ Fixed 114 out of 2062 criteria (5.53%)

🔧 Fixing evaluation trials...
✅ Fixed 31 out of 421 criteria (7.36%)

📋 Sample codes AFTER fixing:
   * Participant must be ≥ 28 day -> CODE_8730
   * Body weight ≥ 5 kg at the ti -> CODE_8239
   * Documented TMA on/after 01 J -> CODE_4891

✅ Saved 149 fixed training trials to ./processed_data/1000_trials//structured_clinical_trials.json
✅ Saved 38 fixed evaluation trials to ./processed_data/1000_trials//structured_clinical_trials_eval.json
✅ Also saved to current directory
